# Fig. 2 Generalization Sweep: Partial Quickcheck

This notebook is designed for the `nf_generalize_fig2` sweep while the full array is still running or partially blocked by maintenance.

It does three things:

1. audits which training tasks have final checkpoints,
2. audits which raw `train_full` sample files exist,
3. plots one-point/P(k)/image diagnostics for whatever samples are already available.

The full SSCD paper-style generalizability plot should still be produced by the offline script after sampling finishes:

```bash
sbatch -A huterer0 scripts/slurm/analyze_nf_generalize_fig2_sscd.sbatch
```


In [ ]:
from __future__ import annotations

import json
import math
import os
import re
import sys
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

PROJECT_CANDIDATES = [
    Path.cwd(),
    Path('/home/jiamingp/diffusion_models_repo'),
    Path('/Users/apple/AI/Diffusion_model'),
]
PROJECT_DIR = next((p for p in PROJECT_CANDIDATES if (p / 'simdiff_eval').exists()), PROJECT_CANDIDATES[0])
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from simdiff_eval.io import as_nchw, load_real_from_config
from simdiff_eval.metrics import batch_power_spectra, field_histogram, power_spectrum_summary

SWEEP_NAME = 'nf_generalize_fig2'
MANIFEST_PATH = PROJECT_DIR / 'local' / SWEEP_NAME / 'manifest.json'
CHECKPOINT_ROOT = Path('/scratch/huterer_root/huterer0/jiamingp/saved_runs') / SWEEP_NAME
SAMPLE_ROOT = PROJECT_DIR / 'results' / SWEEP_NAME / 'samples'
OUTPUT_DIR = PROJECT_DIR / 'results' / SWEEP_NAME / 'quickcheck'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = int(os.environ.get('NF_FIG2_SEED', 123))
SAMPLE_LABEL = os.environ.get('NF_FIG2_SAMPLE_LABEL', 'raw_train_full')
MAX_GENERATED = int(os.environ.get('NF_FIG2_MAX_GENERATED', 512))
MAX_REAL_RAW_CUBES = int(os.environ.get('NF_FIG2_MAX_REAL_RAW_CUBES', 16))
PK_NBINS = int(os.environ.get('NF_FIG2_PK_NBINS', 30))

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 180,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'legend.fontsize': 9,
})

print('project:', PROJECT_DIR)
print('manifest:', MANIFEST_PATH)
print('checkpoint root:', CHECKPOINT_ROOT)
print('sample root:', SAMPLE_ROOT)
print('seed:', SEED)


In [ ]:
rows = json.loads(MANIFEST_PATH.read_text())
for task_id, row in enumerate(rows):
    row['task_id'] = task_id

manifest_df = pd.DataFrame(rows)
display(manifest_df[[
    'task_id', 'run_name', 'arch', 'dataset_tag', 'dataset_size',
    'epochs', 'steps_per_epoch', 'actual_updates', 'checkpoint_every_n_epochs',
    'n_train_simulations'
]])


## Training Checkpoint Audit

A run is treated as complete if its latest `checkpoint-epoch-*` directory is at least `epochs - 1` from the manifest. This works even if the directory name is zero-padded or not.


In [ ]:
EPOCH_RE = re.compile(r'checkpoint-epoch-(\d+)')

def checkpoint_epoch(path: Path) -> int | None:
    m = EPOCH_RE.search(path.name)
    return int(m.group(1)) if m else None

ckpt_rows = []
for row in rows:
    ckpt_dir = Path(row['checkpoint_dir'])
    ckpts = sorted(ckpt_dir.glob('checkpoint-epoch-*')) if ckpt_dir.exists() else []
    parsed = [(checkpoint_epoch(p), p) for p in ckpts]
    parsed = [(e, p) for e, p in parsed if e is not None]
    latest_epoch = max((e for e, _ in parsed), default=None)
    latest_path = next((p for e, p in parsed if e == latest_epoch), None) if latest_epoch is not None else None
    final_epoch = int(row['epochs']) - 1
    ckpt_rows.append({
        'task_id': row['task_id'],
        'run_name': row['run_name'],
        'arch': row['arch'],
        'dataset_size': int(row['dataset_size']),
        'expected_final_epoch': final_epoch,
        'latest_epoch': latest_epoch,
        'final_checkpoint': latest_epoch is not None and latest_epoch >= final_epoch,
        'n_checkpoints': len(parsed),
        'checkpoint_dir': str(ckpt_dir),
        'latest_checkpoint': str(latest_path) if latest_path else '',
    })

ckpt_df = pd.DataFrame(ckpt_rows).sort_values(['arch', 'dataset_size'])
display(ckpt_df)
print('checkpoint audit:', ckpt_df['final_checkpoint'].value_counts(dropna=False).to_dict())

completed_ids = ckpt_df.loc[ckpt_df['final_checkpoint'], 'task_id'].astype(int).tolist()
missing_ids = ckpt_df.loc[~ckpt_df['final_checkpoint'], 'task_id'].astype(int).tolist()
print('completed task ids:', completed_ids)
print('missing/pending task ids:', missing_ids)


## Sample Audit and Commands

Run sampling only for task IDs with completed final checkpoints. If maintenance blocks long training jobs, sampling should still be short, but the scheduler may still reserve nodes depending on the maintenance window.


In [ ]:
def sample_path_for(row: dict[str, Any], seed: int = SEED, sample_label: str = SAMPLE_LABEL) -> Path:
    if row.get('sample_path'):
        return PROJECT_DIR / str(row['sample_path']).format(seed=seed, sample_label=sample_label)
    return SAMPLE_ROOT / f"{row['run_name']}_seed{seed}_{sample_label}.npz"


def npz_n(path: Path) -> int:
    if not path.exists():
        return 0
    try:
        with np.load(path) as data:
            key = 'samples' if 'samples' in data.files else data.files[0]
            return int(data[key].shape[0])
    except Exception as exc:
        print('failed reading', path, exc)
        return 0

sample_rows = []
for row in rows:
    path = sample_path_for(row)
    n = npz_n(path)
    sample_rows.append({
        'task_id': row['task_id'],
        'run_name': row['run_name'],
        'arch': row['arch'],
        'dataset_size': int(row['dataset_size']),
        'final_checkpoint': bool(ckpt_df.set_index('task_id').loc[row['task_id'], 'final_checkpoint']),
        'n_available': n,
        'status': 'ok' if n > 0 else 'missing',
        'sample_path': str(path),
    })

sample_df = pd.DataFrame(sample_rows).sort_values(['arch', 'dataset_size'])
display(sample_df)
print('sample audit:', sample_df['status'].value_counts().to_dict())

to_sample = sample_df[(sample_df['final_checkpoint']) & (sample_df['n_available'] == 0)]['task_id'].astype(int).tolist()
if to_sample:
    ids = ','.join(map(str, to_sample))
    print('Submit samples for completed checkpoints:')
    print(f'cd {PROJECT_DIR}')
    print(f'sbatch -A huterer0 --array={ids}%8 scripts/slurm/sample_nf_generalize_fig2_array.sbatch')
else:
    print('No completed unsampled tasks found.')


## Load Available Samples

This loads only sample files that already exist. Real data is capped with `MAX_REAL_RAW_CUBES` so the notebook stays responsive; the offline SSCD script should be used for the full-reference memorization/generalization score.


In [ ]:
def load_npz_array(path: Path) -> np.ndarray:
    with np.load(path) as data:
        key = 'samples' if 'samples' in data.files else data.files[0]
        return as_nchw(np.asarray(data[key], dtype=np.float32))


def evenly_limit(arr: np.ndarray, limit: int | None) -> np.ndarray:
    arr = np.asarray(arr)
    if limit is None or len(arr) <= limit:
        return arr.copy()
    idx = np.linspace(0, len(arr) - 1, int(limit), dtype=int)
    return arr[idx].copy()

loaded = {}
load_rows = []
for row in rows:
    path = sample_path_for(row)
    if not path.exists():
        continue
    config_path = PROJECT_DIR / row['config']
    generated = evenly_limit(load_npz_array(path), MAX_GENERATED)
    raw_cap = min(int(row.get('n_train_simulations', MAX_REAL_RAW_CUBES)), MAX_REAL_RAW_CUBES)
    real = as_nchw(load_real_from_config(config_path, max_raw_samples=raw_cap))
    loaded[row['run_name']] = {
        'spec': row,
        'real': real,
        'generated': generated,
        'sample_path': path,
    }
    load_rows.append({
        'task_id': row['task_id'],
        'run_name': row['run_name'],
        'arch': row['arch'],
        'dataset_size': int(row['dataset_size']),
        'n_real_loaded': len(real),
        'raw_cap': raw_cap,
        'n_generated': len(generated),
        'sample_path': str(path),
    })

loaded_df = pd.DataFrame(load_rows).sort_values(['arch', 'dataset_size']) if load_rows else pd.DataFrame()
display(loaded_df)
print('loaded sample rows:', len(loaded))


## Available Sample Quality Metrics

These are quick-check metrics using capped real references. They are useful for catching broken runs or broad quality trends, not for final ranking. Lower `hist_l1` and `pk_log10_mae` are better; `std_ratio` and P(k) ratios near 1 are better.


In [ ]:
metric_rows = []
for run_name, bundle in loaded.items():
    row = bundle['spec']
    real = bundle['real']
    generated = bundle['generated']
    rh = field_histogram(real, bins=120)
    gh = field_histogram(generated, bins=120)
    edges = np.asarray(rh['bin_edges'])
    width = float(np.mean(np.diff(edges)))
    hist_l1 = float(np.sum(np.abs(np.asarray(rh['hist']) - np.asarray(gh['hist']))) * width)
    metric_rows.append({
        'task_id': row['task_id'],
        'run_name': run_name,
        'arch': row['arch'],
        'dataset_size': int(row['dataset_size']),
        'n_real': len(real),
        'n_generated': len(generated),
        'hist_l1': hist_l1,
        'generated_std': gh['std'],
        'real_std': rh['std'],
        'std_ratio': gh['std'] / max(rh['std'], 1e-30),
        **power_spectrum_summary(real, generated, nbins=PK_NBINS),
    })

metrics_df = pd.DataFrame(metric_rows)
if len(metrics_df):
    metrics_df = metrics_df.sort_values(['arch', 'dataset_size'])
    out = OUTPUT_DIR / 'nf_generalize_fig2_partial_metrics.csv'
    metrics_df.to_csv(out, index=False)
    print('wrote', out)
    display(metrics_df)
else:
    print('No loaded samples yet.')


In [ ]:
if len(metrics_df):
    for arch, sub in metrics_df.groupby('arch'):
        x = sub['dataset_size'].astype(float)
        fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), sharex=True)
        axes[0].plot(x, sub['hist_l1'], marker='o')
        axes[0].set_ylabel('one-point histogram L1')
        axes[0].set_title('one-point error')
        axes[1].plot(x, sub['pk_log10_mae'], marker='o')
        axes[1].set_ylabel('P(k) log10 MAE')
        axes[1].set_title('P(k) error')
        axes[2].plot(x, sub['std_ratio'], marker='o')
        axes[2].axhline(1.0, color='black', ls=':', lw=1)
        axes[2].set_ylabel('generated std / real std')
        axes[2].set_title('field amplitude')
        for ax in axes:
            ax.set_xscale('log', base=2)
            ax.set_xlabel('training dataset size N')
            ax.grid(alpha=0.25)
        fig.suptitle(f'{arch}: available Fig.2 quick-check metrics')
        fig.tight_layout(rect=(0, 0, 1, 0.92))
        out = OUTPUT_DIR / f'nf_generalize_fig2_{arch}_partial_metrics.png'
        fig.savefig(out, dpi=180, bbox_inches='tight')
        print('wrote', out)
        plt.show()


## Real vs Generated Image Panels

For each available run, the top row is a capped real reference slice and the bottom row is a generated sample. This is the fastest way to see whether the low-N models are producing off-manifold artifacts, memorized-looking structures, or plausible fields.


In [ ]:
if loaded:
    for arch in sorted({bundle['spec']['arch'] for bundle in loaded.values()}):
        bundles = [b for b in loaded.values() if b['spec']['arch'] == arch]
        bundles = sorted(bundles, key=lambda b: int(b['spec']['dataset_size']))
        n = len(bundles)
        fig, axes = plt.subplots(2, n, figsize=(max(3*n, 8), 5.4), squeeze=False)
        for col, bundle in enumerate(bundles):
            row = bundle['spec']
            real = bundle['real']
            generated = bundle['generated']
            vmin = float(np.nanquantile(real, 0.005))
            vmax = float(np.nanquantile(real, 0.995))
            axes[0, col].imshow(real[0, 0], cmap='viridis', vmin=vmin, vmax=vmax)
            axes[0, col].set_title(f"N={row['dataset_size']} real")
            axes[1, col].imshow(generated[0, 0], cmap='viridis', vmin=vmin, vmax=vmax)
            axes[1, col].set_title('generated')
            for ax in axes[:, col]:
                ax.set_xticks([])
                ax.set_yticks([])
        fig.suptitle(f'{arch}: available real vs generated examples')
        fig.tight_layout(rect=(0, 0, 1, 0.92))
        out = OUTPUT_DIR / f'nf_generalize_fig2_{arch}_partial_images.png'
        fig.savefig(out, dpi=180, bbox_inches='tight')
        print('wrote', out)
        plt.show()
else:
    print('No samples available to plot yet.')


## After All Samples Exist

Run the offline SSCD analysis for the paper-style Fig. 2 metric:

```bash
cd /home/jiamingp/diffusion_models_repo
sbatch -A huterer0 scripts/slurm/analyze_nf_generalize_fig2_sscd.sbatch
```

If the maintenance reservation is the only issue for the remaining training tasks, you can resubmit them with a shorter walltime that ends before the reservation. The previously completed `u128` tasks took about 10 hours, so `--time=12:00:00` is a reasonable override when there is at least a 12-hour window before maintenance.
